# Second Stage — Person Embedder (training)

Training procedure for the ResNet-50 / BNNeck embedder used in `main.ipynb`. Training is disabled by default (`do_train=False`); set it to `True` to retrain.

#### Library imports

Embedder imports: `torchvision` ResNet-50, `torchinfo` for the model summary, `wandb` for logging, and the shared helpers from `utils.py`.

In [1]:
import random
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchinfo
import wandb

sys.path.append("/kaggle/input/datasets/alessandrotirelli/person-search-utility")
from pathlib import Path
from typing import Callable, List, Optional, Tuple

from kaggle_secrets import UserSecretsClient
from PIL import Image
from scipy.io import loadmat
from torch.optim import Optimizer, lr_scheduler
from torch.utils.data import DataLoader, Dataset, Sampler
from torch.utils.data.dataloader import default_collate
from torchvision.models import resnet50
from torchvision.transforms import v2
from tqdm.notebook import tqdm
from utils import (crop_detections_from_frame, fix_random,
                   parse_annotations_file, set_requires_grad_for_layer)

#### Parameters definition

Embedder and training hyperparameters. Batches follow a **P×K** scheme (`num_ids_per_batch` × `num_instances`) so every batch contains valid triplets; `label_smoothing` regularizes the auxiliary ID loss. `do_train=False` keeps the notebook runnable.

In [2]:
do_train = False

seed = 42 # random seed

detection_size = (128, 256) # widely used measures in the literature

# model hyperparameters
emb_size = 2048

# training hyperparameters
num_workers = 4
log_interval = 5
num_epochs = 300
num_instances = 8
num_ids_per_batch = 16
batch_size = num_ids_per_batch * num_instances
lr = 1e-3
margin = 0.5
label_smoothing = 0.1  # smoothing factor for ID loss; 0.0 = standard CE


run_name = f"resnet50_bnneck_pk{num_ids_per_batch}x{num_instances}_emb{emb_size}_ep{num_epochs}_idloss"

wandb_project = "Second-Stage-PWR-Embedder"
data_root = Path("/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild") 
root_ckpts = Path("/kaggle/working/ckpts")


#### GPU Initialization

Use CUDA when available, otherwise fall back to CPU.

In [3]:
device = "cpu"
if torch.cuda.is_available():
  print("All good, a Gpu is available.")
  device = torch.device("cuda:0")
else:
  print("Please set GPU via Edit -> Notebook Settings.")

All good, a Gpu is available.


#### Reproducibility and Deterministic Mode

Seed all RNGs and enable deterministic kernels.

In [4]:
fix_random(seed)

## Dataset and Dataloader

`DetectionDataset` builds identity-labelled pedestrian crops from the train frames; the `PKBatchSampler` draws P identities × K instances per batch. A light random Gaussian blur augments robustness to scale.

In [5]:
class DetectionDataset(Dataset):
    def __init__(
        self,
        data_root: Path,
        split_str: str = "train",
        transforms: Optional[Callable] = None
    ) -> None:

        if split_str != "train":
            raise AssertionError(
                f"{split_str} is not supported in this training-only notebook. Please use 'train'."
            )

        # DetectionDataset stores 2 lists of fixed dimensionality
        self.detections: List[Image.Image] = []
        self.pids: List[int] = []
        self.split_str = split_str
        self.transforms = transforms

        ext_images = "jpg"
        ext_annotations = "jpg.mat"
        path_images = data_root / "frames"
        path_annotations = data_root / "annotations"

        variable_name_split = "img_index_train"
        path_split = data_root / "frame_train.mat"

        # read Matlab file
        mat = loadmat(path_split)
        if variable_name_split not in mat:
            raise AssertionError(
                f"Missing {variable_name_split} in {path_split}."
            )
        sample_indexes_list = [str(elem[0][0]) for elem in mat[variable_name_split]] # in the string format: 'cX_sY_ZZZZZZ'

        # converting Python list to Python set to scale down access cost from O(n) to O(1) in list comprehension
        sample_indexes = set(sample_indexes_list)

        images = sorted([
            path for path in path_images.rglob(f"*.{ext_images}")
            if path.name.removesuffix(f".{ext_images}") in sample_indexes
        ])
        annotations = sorted([
            path for path in path_annotations.rglob(f"*.{ext_annotations}")
            if path.name.removesuffix(f".{ext_annotations}") in sample_indexes
        ])

        if len(images) != len(annotations):
            raise AssertionError(
                f"Images and labels differs in size: {len(images)} vs {len(annotations)}."
            )

        for image_path, annotation_path in tqdm(zip(images, annotations), total=len(images), desc="Training detections extraction"):
            ids, boxes = parse_annotations_file(annotation_path)
            boxes = [[box[0], box[1], box[0] + box[2], box[1] + box[3]] for box in boxes] # convert from whxy to xyxy box format
            image = Image.open(image_path).convert("RGB")
            detections = crop_detections_from_frame(image, boxes, detection_size)

            for pid, det in zip(ids, detections):
                pid = int(pid)
                self.detections.append(det)
                self.pids.append(pid)

        # Build a contiguous 0-based label mapping over valid (pid >= 0) identities.
        # Raw PRW pids are arbitrary integers; CrossEntropyLoss requires labels in [0, num_classes-1].
        valid_pids = sorted(set(p for p in self.pids if p >= 0))
        self.pid2label = {pid: idx for idx, pid in enumerate(valid_pids)}

    def __getitem__(self, idx):
        detection = self.detections[idx]
        if self.transforms is not None:
            detection = self.transforms(detection)

        pid = self.pids[idx]
        # Map raw pid -> contiguous label; unlabelled detections (pid < 0) keep pid as-is
        # (they are excluded by the PK sampler and never reach the loss).
        label = self.pid2label[pid]
        return detection, label

    def __len__(self):
        return len(self.detections)

In [6]:
# Create dataset
train_transform = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.RandomApply([
        v2.GaussianBlur(
            kernel_size=(3, 9),   # randomly picks ODD kernel from this range
            sigma=(0.5, 3.0)      # randomly picks (using uniform sampling) sigma in this range
        )
    ], p=0.5), # added gaussian blur to improve generalization over scale
])

train_dataset = DetectionDataset(
    data_root=data_root,
    split_str="train",
    transforms=train_transform
)
print(f"Number of training samples: {len(train_dataset)}")

Training detections extraction:   0%|          | 0/5704 [00:00<?, ?it/s]

Number of training samples: 18031


In [7]:
def collate_skip_none(batch):
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    return default_collate(batch)

class PKBatchSampler(Sampler):
    """Samples P identities and K instances per identity for each batch."""
    def __init__(
            self, 
            labels: List[int], 
            num_ids_per_batch: int, 
            num_instances: int
    ) -> None:
        self.index_dict = {}
        for idx, pid in enumerate(labels):
            pid = int(pid)
            if pid < 0:
                continue
            self.index_dict.setdefault(pid, []).append(idx)
        self.pids = list(self.index_dict.keys())
        self.num_ids_per_batch = num_ids_per_batch
        self.num_instances = num_instances
        self.batch_size = num_ids_per_batch * num_instances

    def __iter__(self):
        pid_list = self.pids.copy()
        random.shuffle(pid_list)
        batch = []
        for pid in pid_list:
            idxs = self.index_dict[pid]
            if len(idxs) >= self.num_instances:
                sampled = random.sample(idxs, self.num_instances)
            else:
                sampled = list(np.random.choice(idxs, self.num_instances, replace=True))
            batch.extend(sampled)
            if len(batch) == self.batch_size:
                yield batch
                batch = []

    def __len__(self) -> int:
        return len(self.pids) // self.num_ids_per_batch


train_batch_sampler = PKBatchSampler(
    labels=train_dataset.pids,
    num_ids_per_batch=num_ids_per_batch,
    num_instances=num_instances
)
train_dataloader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    pin_memory=True,
    num_workers=num_workers, 
    collate_fn=collate_skip_none
)

## Embedder Model Definition

ResNet-50 backbone + 2048-d projection + **BNNeck**, with a training-only **ID classifier** head. The forward pass returns the pre-BNNeck embedding (for the ID loss), the post-BNNeck descriptor (for the triplet loss and retrieval), and the ID logits.

In [8]:
class Embedder(nn.Module):
    """
        Neural network computing an image embedding.

        emb_size: the size for the embedding.
        num_classes: number of training identities, used for the ID classifier head.
        normalize_emb: if True, normalizes bn_feat with L2 at eval/inference time.
        pretrained_feat_ext: if True, uses a pretrained feature extractor.
        train_feat_ext: if True, trains the feature extractor.

        Forward returns:
          - emb     : pre-BNNeck embedding, supervised by ID loss during training.
          - bn_feat : post-BNNeck embedding, supervised by triplet loss during training
                      and used as the retrieval descriptor at inference.
          - logits  : classifier output, only returned during training (model.training).
    """
    def __init__(self,
                 emb_size: int,
                 num_classes: int,
                 normalize_emb: bool,
                 pretrained_feat_ext: bool,
                 train_feat_ext: bool
        ) -> None:
        
        super().__init__()
        
        self.backbone = resnet50( 
            weights="IMAGENET1K_V2" if pretrained_feat_ext else None
        )
        feat_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        set_requires_grad_for_layer(self.backbone, train_feat_ext)

        self.emb = torch.nn.Linear(feat_dim, emb_size, bias=False)
        self.bnneck = torch.nn.BatchNorm1d(emb_size)
        self.bnneck.bias.requires_grad_(False)

        # ID classifier: operates on emb (pre-BN), no bias as BNNeck already
        # centres the features. Weight initialised with normal(0, 0.001).
        self.classifier = nn.Linear(emb_size, num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.normalize_emb = normalize_emb

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.emb(feat)          # pre-BNNeck  -> ID loss target
        bn_feat = self.bnneck(emb)    # post-BNNeck -> triplet loss target + retrieval descriptor

        if self.normalize_emb:
            bn_feat = F.normalize(bn_feat)

        if self.training:
            logits = self.classifier(emb)  # ID loss: classify on pre-BN features
            return emb, bn_feat, logits

        return bn_feat  # inference: only the retrieval descriptor


In [9]:
num_classes = len(train_dataset.pid2label)  # contiguous label count, consistent with pid2label mapping
print(f"Number of training identities: {num_classes}")

embedder = Embedder(
    emb_size=emb_size,
    num_classes=num_classes,
    normalize_emb=True,
    pretrained_feat_ext=True,
    train_feat_ext=True
).to(device)

torchinfo.summary(
    embedder,
    input_size=(1, 3, detection_size[1], detection_size[0]),
    mode="eval"
)


Number of training identities: 483
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 214MB/s]


Layer (type:depth-idx)                        Output Shape              Param #
Embedder                                      [1, 2048]                 989,184
├─ResNet: 1-1                                 [1, 2048]                 --
│    └─Conv2d: 2-1                            [1, 64, 128, 64]          9,408
│    └─BatchNorm2d: 2-2                       [1, 64, 128, 64]          128
│    └─ReLU: 2-3                              [1, 64, 128, 64]          --
│    └─MaxPool2d: 2-4                         [1, 64, 64, 32]           --
│    └─Sequential: 2-5                        [1, 256, 64, 32]          --
│    │    └─Bottleneck: 3-1                   [1, 256, 64, 32]          75,008
│    │    └─Bottleneck: 3-2                   [1, 256, 64, 32]          70,400
│    │    └─Bottleneck: 3-3                   [1, 256, 64, 32]          70,400
│    └─Sequential: 2-6                        [1, 512, 32, 16]          --
│    │    └─Bottleneck: 3-4                   [1, 512, 32, 16]          37

### Utility function: batch-hard triplet loss

The **Triplet Loss**, implemented in the next code section, given an *anchor* $A$, a *positive* $P$ and a *negative* $N$ is given by the formula: $\mathcal{L}=\max\{0, d(A,P) - d(A,N) + m\}$, where the margin $m$ is an hyperparameter used to control the separation between differently labelled groups of data.

In [10]:
def batch_hard_triplet_loss(
    embeddings: torch.Tensor,
    ids: torch.Tensor,
    margin: float
) -> Tuple[torch.Tensor, int]:
    """Computes the semi-hard triplet loss for a batch of embeddings."""
    if embeddings.size(dim=0) < 2: # corner case
        return embeddings.sum() * 0.0, 0

    labels = ids.view(-1)
    dist = 1.0 - embeddings @ embeddings.t() # cosine (dis)similarity
    diag = torch.eye(labels.size(0), device=labels.device, dtype=torch.bool)

    same_label = labels.unsqueeze(dim=0) == labels.unsqueeze(dim=1)
    same_label = same_label & ~diag # removing the diagonal to prevent positive the sample to coincide with the anchor sample

    if not torch.any(same_label):
        return embeddings.sum() * 0.0, 0

    dist_pos = dist.clone()
    dist_pos[~same_label] = -float("inf")
    hardest_pos, _ = dist_pos.max(dim=1)

    valid_pos = hardest_pos != -float("inf")
    if not torch.any(valid_pos):
        return embeddings.sum() * 0.0, 0

    dist_neg = dist.clone()
    dist_neg[same_label | diag] = float("inf")

    semi_mask = (
        valid_pos.unsqueeze(1)
        & (dist_neg > hardest_pos.unsqueeze(1))
        & (dist_neg < (hardest_pos.unsqueeze(1) + margin))
    )
    semi_neg = dist_neg.clone()
    semi_neg[~semi_mask] = float("inf")
    semi_hard_neg, _ = semi_neg.min(dim=1)

    valid = semi_hard_neg != float("inf")
    if not torch.any(valid):
        return embeddings.sum() * 0.0, 0

    loss = torch.relu(hardest_pos[valid] - semi_hard_neg[valid] + margin)
    return loss.mean(), int(valid.sum().item())

## Training Loop

Each step combines the **batch-hard triplet loss** on the post-BNNeck descriptor with the **ID cross-entropy loss** on the logits; the heads are trained at 10× the backbone learning rate (BoT baseline).

In [11]:
id_criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)


def train_epoch(
    model: nn.Module,
    train_dataloader: DataLoader,
    device: torch.device,
    optimizer: Optimizer,
    scheduler: lr_scheduler,
    epoch: int
) -> float:
    """Trains a neural network for one epoch."""
    loss_train = 0.0
    num_batches = 0

    model.train()
    for idx_batch, batch in enumerate(train_dataloader):
        if batch is None:
            continue

        crops, ids = batch
        crops = crops.to(device)
        ids = ids.to(device)

        _, bn_feat, logits = model(crops)

        # Triplet loss on the post-BNNeck L2-normalised descriptor
        triplet_loss, valid_triplets = batch_hard_triplet_loss(bn_feat, ids, margin)

        # ID loss on the pre-BNNeck embedding
        id_loss = id_criterion(logits, ids)

        loss = triplet_loss + id_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        loss_train += loss.item()
        num_batches += 1

        if log_interval > 0 and idx_batch % log_interval == 0:
            wandb.log({
                "train/lr": scheduler.get_last_lr()[0],
                "train/loss": loss.item(),
                "train/triplet_loss": triplet_loss.item(),
                "train/id_loss": id_loss.item(),
                "train/valid_triplets": valid_triplets,
                "train/epoch": epoch
            })

    # Save the model at the end of each epoch
    root_ckpts.mkdir(exist_ok=True)
    torch.save(model.state_dict(), root_ckpts / f"{run_name}.pt")

    if num_batches == 0:
        return 0.0

    return loss_train / num_batches


def training_loop(
    num_epochs: int,
    optimizer: Optimizer,
    scheduler: lr_scheduler,
    model: nn.Module,
    train_dataloader: DataLoader,
    device: torch.device,
) -> None:
    """Executes the training loop."""
    for epoch in tqdm(range(1, num_epochs + 1), "Epoch"):
        loss_train = train_epoch(
            model,
            train_dataloader,
            device,
            optimizer,
            scheduler,
            epoch
        )

        wandb.log({
            "train/avg_epoch_loss": loss_train,
            "train/epoch": epoch
        })


In [12]:
def train(model: nn.Module,
          lr: float,
          num_epochs: int,
          train_dataloader: DataLoader,
          device: torch.device,
          run_name: str,
          wandb_project: str,) -> None:
    """Executes the training loop."""
    # Logging
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("wandb_api_key") 
    wandb.login(key=wandb_key)
    wandb.init(
        project=wandb_project,
        name=run_name,
        settings=wandb.Settings(_disable_stats=True),
    )

    optimizer = torch.optim.SGD(
        [
            {"params": model.backbone.parameters()},
            # heads trained at 10× the backbone lr, consistent with BoT baseline
            {"params": model.emb.parameters(),        "lr": 10 * lr},
            {"params": model.bnneck.parameters(),     "lr": 10 * lr},
            {"params": model.classifier.parameters(), "lr": 10 * lr},
        ], lr=lr, momentum=0.9, weight_decay=5e-4)

    scheduler = lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(train_dataloader)*num_epochs, eta_min=1e-6
    )
    
    training_loop(
        num_epochs,
        optimizer,
        scheduler,
        model,
        train_dataloader,
        device,
    )

    wandb.finish()


## Train Step

Runs only when `do_train=True`; otherwise skipped, since the released checkpoint is loaded in `main.ipynb`.

In [13]:
if do_train:
    train(
        embedder,
        lr,
        num_epochs,
        train_dataloader,
        device,
        run_name,
        wandb_project
    )
else:
    print("Training skip.")

Training skip.
